# 08 · Model comparison

Every model with saved out-of-fold predictions (`outputs/oof/<name>.parquet`), scored on exactly the same
596,953 practice questions and the same 5 folds, straight from its saved file. Nothing is trained here.
Duong's GRU / Transformer are added to `MODELS` once he shares their prediction files.

In [1]:
import pandas as pd

from eventflow.config import EXPERIMENT_LOG, ID_COL
from eventflow.data import load_train, make_train_examples
from eventflow.validation import PROBA_COLS, add_folds, evaluate, load_oof

train = load_train()
examples = add_folds(make_train_examples(train))
print(f"{len(examples):,} examples from {examples[ID_COL].nunique():,} machines")

596,953 examples from 6,500 machines


In [2]:
# Every model with saved out-of-fold predictions (outputs/oof/<name>.parquet).
# Add Duong's GRU / Transformer here once he shares their files.
MODELS = {
    "uniform": "Uniform",
    "class_prior": "Class prior",
    "last1_table": "Last-1 table (test-like cuts)",
    "last1_table_all": "Last-1 table (all events)",
    "current_row_lr": "Current-row LR",
    "last3_table": "Last-3 table",
    "lgbm_all": "LightGBM, 6 groups",
    "lgbm_4groups": "LightGBM, 4 groups",
}

The saved files don't store `is_mimic`, so this adds it back from `examples`. It also checks that every model has a
prediction for all 596,953 examples. That check is what makes the comparison fair: every model is scored on the
same examples and the same folds.

In [ ]:
#load the chances rows from the 8 models then using the chances rows of the 8 model compare with the correct target then give out logloss for each
def score_per_fold(name):
    """Logloss and mimic on each fold for one model, from its saved predictions."""
    oof = load_oof(name)
    df = oof.merge(examples[[ID_COL, "last_row", "is_mimic"]], on=[ID_COL, "last_row"],
                   how="left", validate="one_to_one")
    assert len(df) == len(examples), f"{name}: {len(df):,} rows, expected {len(examples):,}"
    assert df["is_mimic"].notna().all(), f"{name}: some predictions don't match an example"
    rows = []
    for fold, part in df.groupby("fold"):
        r = evaluate(part, part[PROBA_COLS].to_numpy())
        rows.append({"fold": fold, "logloss": r["logloss"], "mimic": r["logloss_mimic"]})
    return pd.DataFrame(rows).set_index("fold")

Like the ablation table, this compares fold by fold, here against the best model.

In [4]:
scores = {name: score_per_fold(name) for name in MODELS}
best = min(scores, key=lambda n: scores[n]["logloss"].mean())

rows = []
for name, s in scores.items():
    change = s["logloss"] - scores[best]["logloss"]          # one number per fold
    rows.append({"model": MODELS[name], "logloss": s["logloss"].mean(), "std": s["logloss"].std(),
                 "mimic": s["mimic"].mean(), "vs best": change.mean(),
                 "worse than best on": f"{(change > 0).sum()} of {len(change)} folds"})
comparison = pd.DataFrame(rows).sort_values("logloss").reset_index(drop=True)
print("Best model:", MODELS[best])
comparison.round(4)

Best model: LightGBM, 4 groups


,model,logloss,std,mimic,vs best,worse than best on
0,"LightGBM, 4 groups",1.4801,0.0026,1.4688,0.0000,0 of 5 folds
1,"LightGBM, 6 groups",1.4809,0.0026,1.4745,0.0008,5 of 5 folds
2,Last-3 table,1.6440,0.0034,1.6367,0.1640,5 of 5 folds
3,Current-row LR,1.7228,0.0025,1.7147,0.2427,5 of 5 folds
4,Last-1 table (test-like cuts),1.7710,0.0029,1.7665,0.2910,5 of 5 folds
5,Last-1 table (all events),1.7746,0.0028,1.7703,0.2946,5 of 5 folds
6,Class prior,2.1638,0.0042,2.1642,0.6838,5 of 5 folds
7,Uniform,2.6391,0.0000,2.6391,1.1590,5 of 5 folds


Each score should match what you logged in `results.csv`. If one doesn't, that model's prediction file is out of date.

In [5]:
logged = (pd.read_csv(EXPERIMENT_LOG).drop_duplicates("run_id", keep="last")
          .set_index("run_id")["val_logloss"])
for name, s in scores.items():
    run_id = name if name in logged.index else f"baseline_{name}"
    print(f"{name:16s} now {s['logloss'].mean():.5f}   logged {logged[run_id]:.5f}")

uniform          now 2.63906   logged 2.63906
class_prior      now 2.16384   logged 2.16384
last1_table      now 1.77102   logged 1.77102
last1_table_all  now 1.77462   logged 1.77462
current_row_lr   now 1.72278   logged 1.72278
last3_table      now 1.64404   logged 1.64404
lgbm_all         now 1.48090   logged 1.48090
lgbm_4groups     now 1.48006   logged 1.48006


## Notes

- All 8 models are scored on the same 596,953 questions and the same 5 folds, rebuilt from their saved predictions. The scores match `results.csv` exactly.
- **LightGBM with 4 groups is the best model: 1.4801 ± 0.0026** (mimic 1.4688). With all 6 groups it is 1.4809: worse on all 5 folds, by 0.0008. Both models are scored on the same folds, so the fold-by-fold comparison is what counts (not the 0.0026 spread between folds): the 6-group model is slightly but consistently worse. So we keep the 4-group model: a little better, and simpler and faster.
- LightGBM beats the best baseline (last-3 table, 1.6440) by 0.164, on every fold.
- The baselines show what each kind of information is worth: uniform 2.639 → class prior 2.164 → last-event table 1.771 → current-row LR 1.723 → last-3 table 1.644. Knowing the last event is the biggest single step; the last row's sensors and the last 3 events each help further; the engineered history features help most.
- Duong's GRU / Transformer are added to `MODELS` once their predictions are shared. The final model is chosen after that.